# FHIR R4 interoperability

ConMedRL can map de-identified canonical ICU events to FHIR R4 NDJSON. Conformance applies to the exchange resources—not to scaled RL CSV files, tensors, or models.

In [ ]:
from pathlib import Path
import pandas as pd

from ConMedRL.data import export_fhir_r4

cohort = pd.DataFrame([{
    "subject_id": "deidentified-local-patient",
    "stay_id": "deidentified-local-stay",
    "los": 2.0,
}])
observations = pd.DataFrame([
    {"stay_id": "deidentified-local-stay", "time_offset_hours": 1.0,
     "variable": "Heart Rate", "value": 82, "unit": "bpm", "source_id": "local-hr"},
    {"stay_id": "deidentified-local-stay", "time_offset_hours": 2.0,
     "variable": "Blood Pressure Systolic", "value": 118, "unit": "mmHg"},
    {"stay_id": "deidentified-local-stay", "time_offset_hours": 2.0,
     "variable": "Blood Pressure Diastolic", "value": 67, "unit": "mmHg"},
])
procedures = pd.DataFrame([{
    "stay_id": "deidentified-local-stay",
    "time_offset_hours": 30.0,
    "procedure": "extubation",
}])

result = export_fhir_r4(
    cohort,
    observations,
    output_dir=Path("./fhir_r4_example"),
    procedures=procedures,
    id_salt="not-for-production-example-salt",
)
print("FHIR resources valid:", result.valid)
print("Validation level:", result.report["validation_level"])
print("Claim scope:", result.report["claim_scope"])
print("Files:", result.paths)

The exporter preserves local codes beside verified LOINC mappings and uses UCUM only when verified. Systolic/diastolic observations at the same time become R4 blood-pressure components. Offset-only sources receive a documented relative-time extension instead of fake calendar dates. Set `FHIRConfig(validator_path="validator_cli.jar")` when building a bundle to add official HL7 `-version 4.0` results to the conformance report.